In [1]:
# Cell 1: Install and import dependencies
%pip -q install nltk tqdm

import os
import re
import json
from pathlib import Path
from collections import Counter, defaultdict

import nltk
from tqdm.auto import tqdm

nltk.download("punkt", quiet=True)

True

In [2]:
# Cell 2: Mount Google Drive and set input/output paths
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/final_project")

# Update this if your file name is different
INPUT_NAME  = "2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json"
OUTPUT_NAME = "2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean_postproc_v2.json"

INPUT_DRIVE_PATH  = DRIVE_DIR / INPUT_NAME
OUTPUT_DRIVE_PATH = DRIVE_DIR / OUTPUT_NAME

LOCAL_INPUT_PATH  = Path("/content") / INPUT_NAME
LOCAL_OUTPUT_PATH = Path("/content") / OUTPUT_NAME

assert INPUT_DRIVE_PATH.exists(), f"Input not found: {INPUT_DRIVE_PATH}"

# Copy from Drive to local (faster processing)
shutil.copy2(INPUT_DRIVE_PATH, LOCAL_INPUT_PATH)
print("Copied to local:", LOCAL_INPUT_PATH)


Mounted at /content/drive
Copied to local: /content/2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json


In [3]:
# Cell 3: Load JSON and validate schema
with LOCAL_INPUT_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

assert isinstance(data, list) and len(data) > 0, "JSON must be a non-empty list."

# Quick schema check on first item
sample = data[0]
for k in ["_id", "titles", "docs", "docs2", "context"]:
    assert k in sample, f"Missing key in examples: {k}"

print("Loaded examples:", len(data))
print("Sample keys:", list(sample.keys()))
print("First example titles/docs/docs2 lens:",
      len(sample["titles"]), len(sample["docs"]), len(sample["docs2"]))


Loaded examples: 1000
Sample keys: ['_id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context', 'titles', 'docs', 'docs2', 'evidences']
First example titles/docs/docs2 lens: 10 10 10


In [4]:
# Cell 4: Define a safer sentence tokenizer and post-processing rules (with empty-parentheses removal)
from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktParameters

# Common abbreviations that often break sentence splitting in Wikipedia-style text
ABBREV_TYPES = {
    "vol", "no", "pp", "p", "fig", "al", "ed", "eds", "jr", "sr",
    "st", "mt", "mr", "mrs", "ms", "dr", "prof", "rev",
    "inc", "ltd", "co", "corp", "dept", "univ"
}

punkt_params = PunktParameters()
punkt_params.abbrev_types = set(ABBREV_TYPES)
SENT_TOKENIZER = PunktSentenceTokenizer(punkt_params)

# --- Utilities
_RE_WS = re.compile(r"\s+")
_RE_PUNCT_ONLY = re.compile(r"^[\W_]+$", flags=re.UNICODE)

# Entire-string empty wrappers (already handled before, keep them)
_RE_EMPTY_PARENS = re.compile(r"^\(\s*\)$")
_RE_EMPTY_BRACKETS = re.compile(r"^\[\s*\]$")
_RE_EMPTY_BRACES = re.compile(r"^\{\s*\}$")

# Inline empty parentheses inside a sentence: "Name () is ..." or "Name ( ) is ..."
_RE_INLINE_EMPTY_PARENS = re.compile(r"\(\s*\)")

# Optional: also remove inline empty brackets/braces if they appear
_RE_INLINE_EMPTY_BRACKETS = re.compile(r"\[\s*\]")
_RE_INLINE_EMPTY_BRACES = re.compile(r"\{\s*\}")

# URL-like tokens with no spaces (incl. domains without http)
_RE_URLISH = re.compile(
    r"^(?:https?://|www\.)\S+$|^[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?:/\S+)?$"
)

# If a previous sentence ends with one of these, it is likely NOT a true boundary
_RE_PREV_ABBREV_END = re.compile(
    r"(?:\b(?:vol|no|pp|p|fig|ed|eds|jr|sr|st|mt|mr|mrs|ms|dr|prof|rev|inc|ltd|co|corp)\.)\s*$",
    flags=re.IGNORECASE
)

# Small fragments like "2, pp." or "45-77." or "8–9."
_RE_NUM_FRAGMENT = re.compile(r"^(?:[IVXLCDM]+|\d+)(?:[\w\W]{0,25})$", flags=re.IGNORECASE)
_RE_PAGE_RANGE = re.compile(r"^\d+\s*[–-]\s*\d+\.?$")

def remove_inline_empty_wrappers(s: str) -> str:
    """Remove inline empty (), [], {} occurring anywhere in the text."""
    if not isinstance(s, str) or not s:
        return ""
    s = _RE_INLINE_EMPTY_PARENS.sub(" ", s)
    s = _RE_INLINE_EMPTY_BRACKETS.sub(" ", s)
    s = _RE_INLINE_EMPTY_BRACES.sub(" ", s)
    return s

def _norm_ws(s: str) -> str:
    """Normalize whitespace and remove inline empty wrappers like () or ( )."""
    if not isinstance(s, str) or not s:
        return ""
    s = s.replace("\u00a0", " ")
    s = remove_inline_empty_wrappers(s)
    s = _RE_WS.sub(" ", s).strip()
    return s

def is_junk_sentence(s: str) -> bool:
    """Return True for punctuation-only, empty bracket/paren tokens, and URL-only tokens."""
    if not s:
        return True
    t = s.strip()
    if not t:
        return True
    if _RE_EMPTY_PARENS.fullmatch(t) or _RE_EMPTY_BRACKETS.fullmatch(t) or _RE_EMPTY_BRACES.fullmatch(t):
        return True
    if _RE_PUNCT_ONLY.fullmatch(t):
        return True
    if _RE_URLISH.fullmatch(t) and (" " not in t):
        return True
    if t.lower().startswith("category:"):
        return True
    return False

def merge_text(prev: str, cur: str) -> str:
    """Merge two sentence fragments with minimal punctuation artifacts."""
    prev = prev.rstrip()
    cur = cur.lstrip()

    # If current starts with punctuation, attach without extra space
    if cur and cur[0] in ",.;:)]}":
        out = prev + cur
    else:
        out = prev + " " + cur

    return _norm_ws(out)

def should_merge_with_prev(prev: str, cur: str) -> bool:
    """Heuristics to merge fragments that should not be standalone sentences."""
    if not prev:
        return False
    if not cur:
        return True

    cur_stripped = cur.strip()

    # If current starts with lowercase, it's almost certainly a continuation
    if re.match(r"^[a-z]", cur_stripped):
        return True

    # If previous ends with abbreviation like "Vol." or "pp.", likely continuation
    if _RE_PREV_ABBREV_END.search(prev):
        return True

    # If current looks like a pure numeric fragment / page range / roman fragment
    # and previous is short-ish or ends with comma/abbrev, merge.
    if _RE_PAGE_RANGE.fullmatch(cur_stripped) or _RE_NUM_FRAGMENT.fullmatch(cur_stripped):
        if prev.endswith(",") or _RE_PREV_ABBREV_END.search(prev) or len(prev) <= 60:
            return True

    # Single-letter initial like "R." should not be alone
    if re.fullmatch(r"[A-Z]\.", cur_stripped):
        return True

    # Very short fragments that end with "." and have no obvious verb/content
    if len(cur_stripped) <= 12 and cur_stripped.endswith(".") and cur_stripped.count(" ") <= 1:
        return True

    return False

def tokenize_and_clean(doc_text: str) -> list[str]:
    """Tokenize doc_text into sentences, then drop junk + merge fragments."""
    if not isinstance(doc_text, str) or not doc_text.strip():
        return []

    # Remove inline empty wrappers early (important for patterns like 'Name () is ...')
    text = doc_text.replace("\r\n", "\n").replace("\r", "\n").strip()
    text = remove_inline_empty_wrappers(text)

    # Tokenize paragraph-wise to reduce cross-paragraph weird merges
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]

    raw_sents = []
    for p in paragraphs:
        p = remove_inline_empty_wrappers(p)
        try:
            sents = SENT_TOKENIZER.tokenize(p)
        except Exception:
            sents = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])", p)
        raw_sents.extend(sents)

    # Normalize whitespace and remove obvious junk
    cleaned = []
    for s in raw_sents:
        s2 = _norm_ws(s)
        if is_junk_sentence(s2):
            continue
        cleaned.append(s2)

    # Merge pass
    merged = []
    for s in cleaned:
        if not merged:
            merged.append(s)
            continue
        if is_junk_sentence(s):
            continue
        if should_merge_with_prev(merged[-1], s):
            merged[-1] = merge_text(merged[-1], s)
        else:
            merged.append(s)

    # Final drop: remove anything that became junk after merging
    merged2 = []
    for s in merged:
        s2 = _norm_ws(s)
        if not is_junk_sentence(s2):
            merged2.append(s2)

    return merged2

In [5]:
# Cell 5: Process all examples and preserve paragraph structure in docs
# - docs2 stays a flat list of sentences
# - docs is rebuilt paragraph-by-paragraph using original paragraph breaks

stats = Counter()
examples_with_changes = 0

def clean_sentence_list(raw_sents: list[str]) -> list[str]:
    """Normalize + drop junk + merge fragments inside a paragraph."""
    cleaned = []
    for s in raw_sents:
        s2 = _norm_ws(s)
        if is_junk_sentence(s2):
            continue
        cleaned.append(s2)

    merged = []
    for s in cleaned:
        if not merged:
            merged.append(s)
            continue
        if should_merge_with_prev(merged[-1], s):
            merged[-1] = merge_text(merged[-1], s)
        else:
            merged.append(s)

    merged2 = []
    for s in merged:
        s2 = _norm_ws(s)
        if not is_junk_sentence(s2):
            merged2.append(s2)

    return merged2

def tokenize_clean_keep_paragraphs(doc_text: str) -> tuple[list[list[str]], list[str]]:
    """
    Returns:
      paras_sents: list of paragraphs, each paragraph is list of sentences
      flat_sents : flattened sentences (docs2)
    """
    if not isinstance(doc_text, str) or not doc_text.strip():
        return [], []

    text = doc_text.replace("\r\n", "\n").replace("\r", "\n").strip()
    text = remove_inline_empty_wrappers(text)

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]

    paras_sents = []
    for p in paragraphs:
        p = remove_inline_empty_wrappers(p)
        try:
            raw = SENT_TOKENIZER.tokenize(p)
        except Exception:
            raw = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])", p)

        sents = clean_sentence_list(raw)
        if sents:
            paras_sents.append(sents)

    flat = [s for para in paras_sents for s in para]
    return paras_sents, flat

for ex in tqdm(data, desc="Post-processing JSON (keep paragraphs in docs)"):
    titles = ex.get("titles", [])
    docs   = ex.get("docs", [])
    docs2  = ex.get("docs2", [])

    if not (isinstance(titles, list) and isinstance(docs, list) and isinstance(docs2, list)):
        stats["bad_schema"] += 1
        continue

    if not (len(titles) == len(docs) == len(docs2)):
        stats["alignment_issue"] += 1
        continue

    new_docs = []
    new_docs2 = []
    changed_any = False

    for i in range(len(titles)):
        doc_text  = docs[i] if isinstance(docs[i], str) else ""
        old_sents = docs2[i] if isinstance(docs2[i], list) else []

        paras_sents, flat_sents = tokenize_clean_keep_paragraphs(doc_text)

        # Fallback: if docs text is empty but docs2 existed, clean docs2 and treat as single paragraph
        if not flat_sents and isinstance(old_sents, list) and old_sents:
            tmp = []
            for s in old_sents:
                if not isinstance(s, str):
                    continue
                s2 = _norm_ws(s)
                if is_junk_sentence(s2):
                    continue
                if tmp and should_merge_with_prev(tmp[-1], s2):
                    tmp[-1] = merge_text(tmp[-1], s2)
                else:
                    tmp.append(s2)
            flat_sents = [x for x in tmp if not is_junk_sentence(x)]
            paras_sents = [flat_sents] if flat_sents else []

        # Rebuild docs with paragraph boundaries preserved
        rebuilt_doc = "\n\n".join(" ".join(para) for para in paras_sents) if paras_sents else ""

        # Track changes (compare under whitespace normalization so paragraph breaks don't count as "difference")
        old_join = _norm_ws(" ".join(_norm_ws(x) for x in old_sents if isinstance(x, str))) if old_sents else ""
        if _norm_ws(rebuilt_doc) != _norm_ws(doc_text):
            changed_any = True
        if old_sents and _norm_ws(rebuilt_doc) != old_join:
            changed_any = True

        new_docs.append(rebuilt_doc)
        new_docs2.append(flat_sents)

        stats["sent_before"] += len([x for x in old_sents if isinstance(x, str)])
        stats["sent_after"]  += len(flat_sents)

    ex["docs"] = new_docs
    ex["docs2"] = new_docs2

    if changed_any:
        examples_with_changes += 1

stats["examples_changed"] = examples_with_changes
stats["examples_total"] = len(data)

print("Done.")
print("Stats:", dict(stats))

Post-processing JSON (keep paragraphs in docs):   0%|          | 0/1000 [00:00<?, ?it/s]

Done.
Stats: {'sent_before': 232330, 'sent_after': 223466, 'examples_changed': 491, 'examples_total': 1000}


In [6]:
# Cell 6: Validate strict docs/docs2 consistency + dataset context containment
import unicodedata

def norm_containment(s: str) -> str:
    s = str(s)
    s = s.replace("\u00a0", " ")
    s = unicodedata.normalize("NFKC", s)
    s = s.casefold()
    s = re.sub(r"[\W_]+", "", s, flags=re.UNICODE)
    return s

def get_ds_titles_and_sents_from_context(ex: dict) -> dict[str, list[str]]:
    """Return map: title -> [sentences] from Hotpot-like 'context'."""
    ctx = ex.get("context")
    out = {}
    if not isinstance(ctx, list):
        return out
    for item in ctx:
        if isinstance(item, list) and len(item) == 2 and isinstance(item[0], str) and isinstance(item[1], list):
            out[item[0]] = [s for s in item[1] if isinstance(s, str)]
    return out

strict_mismatch = 0
contain_mismatch = 0
first_bad = []

for ex in tqdm(data, desc="Validating"):
    titles = ex.get("titles", [])
    docs   = ex.get("docs", [])
    docs2  = ex.get("docs2", [])

    if not (isinstance(titles, list) and isinstance(docs, list) and isinstance(docs2, list)):
        continue
    if not (len(titles) == len(docs) == len(docs2)):
        continue

    # (1) Strict consistency: docs[i] == " ".join(docs2[i]) after whitespace normalization
    for i in range(len(titles)):
        d = docs[i] if isinstance(docs[i], str) else ""
        sents = docs2[i] if isinstance(docs2[i], list) else []
        j = _norm_ws(" ".join([_norm_ws(x) for x in sents if isinstance(x, str)]))
        if _norm_ws(d) != j:
            strict_mismatch += 1
            if len(first_bad) < 5:
                first_bad.append(("strict", ex.get("_id"), titles[i], d[:120], j[:120]))

    # (2) Dataset context sentences containment
    ds_map = get_ds_titles_and_sents_from_context(ex)
    title_to_idx = {t: i for i, t in enumerate(titles) if isinstance(t, str)}

    for t, ds_sents in ds_map.items():
        if t not in title_to_idx:
            continue
        i = title_to_idx[t]
        blob = norm_containment(" ".join(docs2[i]))  # normalized joined docs2
        for s in ds_sents:
            sn = norm_containment(s)
            if sn and sn not in blob:
                contain_mismatch += 1
                if len(first_bad) < 10:
                    first_bad.append(("contain", ex.get("_id"), t, s[:180], "NOT_FOUND"))
                break

print("Strict docs/docs2 mismatches:", strict_mismatch)
print("Context containment mismatches:", contain_mismatch)

if first_bad:
    print("\nFirst issues (up to 10):")
    for row in first_bad:
        print(row)


Validating:   0%|          | 0/1000 [00:00<?, ?it/s]

Strict docs/docs2 mismatches: 0
Context containment mismatches: 0


In [8]:
# Cell 6.5: Pretty-print 1 random example from the NEW JSON (prefer one that changed)

import random
from pprint import pprint

# Try to restrict sampling to examples that changed (silent check; no before/after printing)
changed_indices = None
try:
    with LOCAL_INPUT_PATH.open("r", encoding="utf-8") as f:
        data_before = json.load(f)

    if isinstance(data_before, list) and len(data_before) == len(data):
        tmp = []
        for i in range(len(data)):
            b = data_before[i]
            a = data[i]
            if b.get("docs") != a.get("docs") or b.get("docs2") != a.get("docs2"):
                tmp.append(i)
        changed_indices = tmp
except Exception:
    changed_indices = None

# Pick one index
if changed_indices and len(changed_indices) > 0:
    idx = random.choice(changed_indices)
    print(f"[Info] Sampling from CHANGED examples only. Changed pool size: {len(changed_indices)}")
else:
    idx = random.randrange(len(data))
    print("[Info] Could not (or did not) restrict to changed-only; sampling from all examples.")

ex = data[idx]

print("\n" + "=" * 24, "Random Sample", "=" * 24)
print("index:", idx)
print("_id:", ex.get("_id"))
print("type:", ex.get("type"))
print("question:", ex.get("question"))
print("-" * 70)
pprint(ex, width=140, sort_dicts=False)


[Info] Sampling from CHANGED examples only. Changed pool size: 978

======================== Random Sample ========================
index: 273
_id: b583abd20baf11ebab90acde48001122
type: inference
question: Who is the paternal grandfather of Sultan Al-Dawla?
----------------------------------------------------------------------
{'_id': 'b583abd20baf11ebab90acde48001122',
 'question': 'Who is the paternal grandfather of Sultan Al-Dawla?',
 'answer': "'Adud al-Dawla",
 'type': 'inference',
 'level': None,
 'supporting_facts': [['Sultan al-Dawla', 1], ['Baha al-Dawla', 3]],
 'context': [["'Adud al-Dawla",
              ['Fannā( Panāh) Khusraw, better known by his laqab of ʿ Aḍud al- Dawla(" Pillar of the[ Abbasid] Dynasty")',
               '( September 24, 936 – March 26, 983) was an emir of the Buyid dynasty, ruling from 949 to 983, and at his height of power '
               'ruling an empire stretching from Makran as far to Yemen and the shores of the Mediterranean Sea.',
            

In [9]:
# Cell 7: Save the processed JSON with a new name
with LOCAL_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)  # no indent -> smaller file

# Copy back to Drive
import shutil
shutil.copy2(LOCAL_OUTPUT_PATH, OUTPUT_DRIVE_PATH)

print("Saved local:", LOCAL_OUTPUT_PATH)
print("Saved to Drive:", OUTPUT_DRIVE_PATH)

Saved local: /content/2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean_postproc_v2.json
Saved to Drive: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean_postproc_v2.json
